# NFL Big Data Bowl 2026 - Model Training & Submission

This notebook trains a player trajectory prediction model and generates a submission file for the leaderboard.

## Strategy:
1. **Baseline Model**: Linear extrapolation with physics constraints
2. **Enhanced Model**: LSTM-based sequence prediction with contextual features
3. **Submission**: Generate predictions in required format

## 1. Import Libraries and Setup

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import joblib

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load and Prepare Training Data

In [8]:
# Load training data (using first few weeks for faster training)
train_folder = Path(r'c:\nfl-big-data-bowl-2026-prediction\train')

# Load first 3 weeks for training (can expand later)
weeks_to_use = ['w01', 'w02', 'w03']
input_dfs = []
output_dfs = []

print("Loading training data...")
for week in weeks_to_use:
    input_file = train_folder / f"input_2023_{week}.csv"
    output_file = train_folder / f"output_2023_{week}.csv"
    
    if input_file.exists() and output_file.exists():
        input_df = pd.read_csv(input_file)
        output_df = pd.read_csv(output_file)
        
        input_dfs.append(input_df)
        output_dfs.append(output_df)
        print(f"Loaded {week}: {len(input_df)} input rows, {len(output_df)} output rows")

# Combine all weeks
train_input = pd.concat(input_dfs, ignore_index=True)
train_output = pd.concat(output_dfs, ignore_index=True)

print(f"\nTotal training data:")
print(f"Input: {len(train_input)} rows")
print(f"Output: {len(train_output)} rows")
print(f"Unique plays: {train_input[['game_id', 'play_id']].drop_duplicates().shape[0]}")

# Basic data info
print(f"\nInput columns: {list(train_input.columns)}")
print(f"Output columns: {list(train_output.columns)}")

Loading training data...
Loaded w01: 285714 input rows, 32088 output rows
Loaded w02: 288586 input rows, 32180 output rows
Loaded w03: 297757 input rows, 36080 output rows

Total training data:
Input: 872057 rows
Output: 100348 rows
Unique plays: 2573

Input columns: ['game_id', 'play_id', 'player_to_predict', 'nfl_id', 'frame_id', 'play_direction', 'absolute_yardline_number', 'player_name', 'player_height', 'player_weight', 'player_birth_date', 'player_position', 'player_side', 'player_role', 'x', 'y', 's', 'a', 'dir', 'o', 'num_frames_output', 'ball_land_x', 'ball_land_y']
Output columns: ['game_id', 'play_id', 'nfl_id', 'frame_id', 'x', 'y']


## 3. Feature Engineering

In [9]:
def create_features(df):
    """Create features for training"""
    df = df.copy()
    
    # Velocity components
    df['vx'] = df['s'] * np.cos(np.radians(df['dir']))
    df['vy'] = df['s'] * np.sin(np.radians(df['dir']))
    
    # Distance to ball landing
    df['dist_to_ball'] = np.sqrt((df['x'] - df['ball_land_x'])**2 + 
                                (df['y'] - df['ball_land_y'])**2)
    
    # Direction to ball landing
    df['dir_to_ball'] = np.degrees(np.arctan2(df['ball_land_y'] - df['y'], 
                                             df['ball_land_x'] - df['x']))
    
    # Field position features
    df['x_normalized'] = df['x'] / 120.0  # Normalize by field length
    df['y_normalized'] = df['y'] / 53.3   # Normalize by field width
    
    # Player side encoding
    df['is_offense'] = (df['player_side'] == 'Offense').astype(int)
    
    # Position encoding (simplified)
    position_encoder = LabelEncoder()
    df['position_encoded'] = position_encoder.fit_transform(df['player_position'])
    
    # Role encoding
    role_encoder = LabelEncoder()
    df['role_encoded'] = role_encoder.fit_transform(df['player_role'])
    
    return df, position_encoder, role_encoder

# Apply feature engineering
print("Creating features...")
train_input_features, pos_encoder, role_encoder = create_features(train_input)

# Select features for training
feature_columns = [
    'x', 'y', 's', 'a', 'dir', 'o', 'vx', 'vy',
    'dist_to_ball', 'dir_to_ball', 'x_normalized', 'y_normalized',
    'is_offense', 'position_encoded', 'role_encoded', 'num_frames_output',
    'ball_land_x', 'ball_land_y', 'frame_id'
]

print(f"Selected {len(feature_columns)} features for training")
print("Features:", feature_columns)

Creating features...
Selected 19 features for training
Features: ['x', 'y', 's', 'a', 'dir', 'o', 'vx', 'vy', 'dist_to_ball', 'dir_to_ball', 'x_normalized', 'y_normalized', 'is_offense', 'position_encoded', 'role_encoded', 'num_frames_output', 'ball_land_x', 'ball_land_y', 'frame_id']


## 4. Prepare Training Dataset

In [10]:
# Create training pairs (input -> output)
def create_training_pairs(input_df, output_df):
    """Create training pairs by matching input and output frames"""
    
    training_pairs = []
    
    # Group by play to process each play separately
    for (game_id, play_id), play_input in input_df.groupby(['game_id', 'play_id']):
        
        # Get corresponding output data
        play_output = output_df[(output_df['game_id'] == game_id) & 
                               (output_df['play_id'] == play_id)]
        
        if len(play_output) == 0:
            continue
            
        # For each player in output, find their input data
        for nfl_id in play_output['nfl_id'].unique():
            
            player_input = play_input[play_input['nfl_id'] == nfl_id]
            player_output = play_output[play_output['nfl_id'] == nfl_id]
            
            if len(player_input) == 0 or len(player_output) == 0:
                continue
                
            # Use last frame from input as features
            last_input_frame = player_input.loc[player_input['frame_id'].idxmax()]
            
            # Create training examples for each output frame
            for _, output_row in player_output.iterrows():
                
                training_example = {
                    'game_id': game_id,
                    'play_id': play_id,
                    'nfl_id': nfl_id,
                    'output_frame': output_row['frame_id'],
                    'target_x': output_row['x'],
                    'target_y': output_row['y']
                }
                
                # Add input features
                for col in feature_columns:
                    if col in last_input_frame.index:
                        training_example[f'input_{col}'] = last_input_frame[col]
                
                training_pairs.append(training_example)
    
    return pd.DataFrame(training_pairs)

# Create training dataset
print("Creating training pairs...")
training_data = create_training_pairs(train_input_features, train_output)

print(f"Created {len(training_data)} training examples")
print(f"Unique plays: {training_data[['game_id', 'play_id']].drop_duplicates().shape[0]}")
print(f"Unique players: {training_data['nfl_id'].nunique()}")

# Show sample
print("\nSample training data:")
print(training_data.head())

Creating training pairs...
Created 100348 training examples
Unique plays: 2573
Unique players: 781

Sample training data:
      game_id  play_id  nfl_id  output_frame  target_x  target_y  input_x  \
0  2023090700      101   46137           1.0     56.22     17.28    55.82   
1  2023090700      101   46137           2.0     56.63     16.88    55.82   
2  2023090700      101   46137           3.0     57.06     16.46    55.82   
3  2023090700      101   46137           4.0     57.48     16.02    55.82   
4  2023090700      101   46137           5.0     57.91     15.56    55.82   

   input_y  input_s  input_a  ...  input_dir_to_ball  input_x_normalized  \
0    17.67     5.34      1.8  ...          -67.41881            0.465167   
1    17.67     5.34      1.8  ...          -67.41881            0.465167   
2    17.67     5.34      1.8  ...          -67.41881            0.465167   
3    17.67     5.34      1.8  ...          -67.41881            0.465167   
4    17.67     5.34      1.8  ...  

## 5. Train Prediction Models

In [11]:
# Prepare features and targets
feature_cols = [f'input_{col}' for col in feature_columns if f'input_{col}' in training_data.columns]
feature_cols.append('output_frame')  # Add frame number as feature

X = training_data[feature_cols].fillna(0)
y_x = training_data['target_x']
y_y = training_data['target_y']

print(f"Training features: {len(feature_cols)}")
print(f"Training samples: {len(X)}")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_val, y_x_train, y_x_val, y_y_train, y_y_val = train_test_split(
    X_scaled, y_x, y_y, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")

# Train models for X and Y coordinates
print("\nTraining X-coordinate model...")
model_x = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_x.fit(X_train, y_x_train)

print("Training Y-coordinate model...")
model_y = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model_y.fit(X_train, y_y_train)

# Evaluate models
y_x_pred = model_x.predict(X_val)
y_y_pred = model_y.predict(X_val)

mse_x = mean_squared_error(y_x_val, y_x_pred)
mse_y = mean_squared_error(y_y_val, y_y_pred)
rmse_x = np.sqrt(mse_x)
rmse_y = np.sqrt(mse_y)

print(f"\nValidation Results:")
print(f"X-coordinate RMSE: {rmse_x:.3f} yards")
print(f"Y-coordinate RMSE: {rmse_y:.3f} yards")
print(f"Combined RMSE: {np.sqrt(mse_x + mse_y):.3f} yards")

# Save models
joblib.dump(model_x, 'model_x.pkl')
joblib.dump(model_y, 'model_y.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(pos_encoder, 'pos_encoder.pkl')
joblib.dump(role_encoder, 'role_encoder.pkl')

print("\nModels saved successfully!")

Training features: 20
Training samples: 100348
Training set: 80278 samples
Validation set: 20070 samples

Training X-coordinate model...
Training Y-coordinate model...

Validation Results:
X-coordinate RMSE: 0.301 yards
Y-coordinate RMSE: 0.300 yards
Combined RMSE: 0.425 yards

Models saved successfully!


## 6. Load Test Data and Generate Predictions

In [6]:
# Load test data
test_input_path = Path(r'c:\nfl-big-data-bowl-2026-prediction\test_input.csv')
test_path = Path(r'c:\nfl-big-data-bowl-2026-prediction\test.csv')

print("Loading test data...")
test_input = pd.read_csv(test_input_path)
test_targets = pd.read_csv(test_path)

print(f"Test input: {len(test_input)} rows")
print(f"Test targets: {len(test_targets)} rows")

# Apply feature engineering to test data
test_input_features, _, _ = create_features(test_input)

def generate_predictions(test_input_df, test_targets_df, model_x, model_y, scaler):
    """Generate predictions for test data"""
    
    predictions = []
    
    # Group test targets by play
    for (game_id, play_id), play_targets in test_targets_df.groupby(['game_id', 'play_id']):
        
        # Get input data for this play
        play_input = test_input_df[(test_input_df['game_id'] == game_id) & 
                                  (test_input_df['play_id'] == play_id)]
        
        if len(play_input) == 0:
            # If no input data, use simple fallback
            for _, target_row in play_targets.iterrows():
                predictions.append({
                    'id': f"{target_row['game_id']}_{target_row['play_id']}_{target_row['nfl_id']}_{target_row['frame_id']}",
                    'x': 50.0,  # Fallback to field center
                    'y': 26.65
                })
            continue
            
        # Process each target prediction
        for _, target_row in play_targets.iterrows():
            
            player_id = target_row['nfl_id']
            target_frame = target_row['frame_id']
            
            # Get player's input data
            player_input = play_input[play_input['nfl_id'] == player_id]
            
            if len(player_input) == 0:
                # Fallback prediction
                pred_x, pred_y = 50.0, 26.65
            else:
                # Use last available frame for prediction
                last_frame = player_input.loc[player_input['frame_id'].idxmax()]
                
                # Prepare features
                feature_values = []
                for col in feature_columns:
                    if col in last_frame.index:
                        feature_values.append(last_frame[col])
                    elif col == 'num_frames_output':
                        feature_values.append(target_frame)  # Use target frame as proxy
                    else:
                        feature_values.append(0)
                
                feature_values.append(target_frame)  # Add output frame
                
                # Scale features
                features_scaled = scaler.transform([feature_values])
                
                # Predict
                pred_x = model_x.predict(features_scaled)[0]
                pred_y = model_y.predict(features_scaled)[0]
                
                # Apply field constraints
                pred_x = np.clip(pred_x, 0, 120)
                pred_y = np.clip(pred_y, 0, 53.3)
            
            # Add to predictions
            predictions.append({
                'id': f"{target_row['game_id']}_{target_row['play_id']}_{target_row['nfl_id']}_{target_row['frame_id']}",
                'x': pred_x,
                'y': pred_y
            })
    
    return pd.DataFrame(predictions)

# Generate predictions
print("Generating predictions...")
submission = generate_predictions(test_input_features, test_targets, model_x, model_y, scaler)

print(f"Generated {len(submission)} predictions")
print("\nSample predictions:")
print(submission.head())

Loading test data...
Test input: 49753 rows
Test targets: 5837 rows
Generating predictions...
Generated 5837 predictions

Sample predictions:
                      id        x        y
0  2024120805_74_54586_1  88.4972  34.3677
1  2024120805_74_54586_2  88.6429  34.3950
2  2024120805_74_54586_3  88.8685  34.4041
3  2024120805_74_54586_4  89.0365  34.5159
4  2024120805_74_54586_5  89.1433  34.6321


## 7. Create Submission File

In [12]:
# Validate submission format
sample_submission = pd.read_csv(r'c:\nfl-big-data-bowl-2026-prediction\sample_submission.csv')

print("Submission validation:")
print(f"Expected columns: {list(sample_submission.columns)}")
print(f"Our columns: {list(submission.columns)}")
print(f"Expected rows: {len(sample_submission)}")
print(f"Our rows: {len(submission)}")

# Check if all required IDs are present
expected_ids = set(sample_submission['id'])
our_ids = set(submission['id'])

missing_ids = expected_ids - our_ids
extra_ids = our_ids - expected_ids

print(f"Missing IDs: {len(missing_ids)}")
print(f"Extra IDs: {len(extra_ids)}")

# Fill in any missing predictions with fallback values
if missing_ids:
    print(f"Adding {len(missing_ids)} missing predictions...")
    missing_predictions = []
    for missing_id in missing_ids:
        missing_predictions.append({
            'id': missing_id,
            'x': 50.0,  # Field center
            'y': 26.65
        })
    
    missing_df = pd.DataFrame(missing_predictions)
    submission = pd.concat([submission, missing_df], ignore_index=True)

# Remove extra predictions
if extra_ids:
    print(f"Removing {len(extra_ids)} extra predictions...")
    submission = submission[submission['id'].isin(expected_ids)]

# Ensure correct order
submission = submission.merge(sample_submission[['id']], on='id', how='right')
submission = submission.sort_values('id').reset_index(drop=True)

# Final validation
print(f"\nFinal submission shape: {submission.shape}")
print(f"Expected shape: {sample_submission.shape}")
print(f"Shapes match: {submission.shape == sample_submission.shape}")

# Check for missing values
print(f"Missing values: {submission.isnull().sum().sum()}")

# Fill any remaining missing values
submission = submission.fillna({'x': 50.0, 'y': 26.65})

# Save submission file
submission_path = r'c:\nfl-big-data-bowl-2026-prediction\submission.csv'
submission.to_csv(submission_path, index=False)

print(f"\nSubmission file saved to: {submission_path}")
print(f"File size: {Path(submission_path).stat().st_size / 1024:.1f} KB")

# Show submission summary
print(f"\nSubmission Summary:")
print(f"Total predictions: {len(submission)}")
print(f"X coordinate range: {submission['x'].min():.1f} to {submission['x'].max():.1f}")
print(f"Y coordinate range: {submission['y'].min():.1f} to {submission['y'].max():.1f}")
print(f"Average X: {submission['x'].mean():.1f}")
print(f"Average Y: {submission['y'].mean():.1f}")

print("\n" + "="*50)
print("🏈 SUBMISSION READY FOR LEADERBOARD! 🏈")
print("="*50)
print(f"File: {submission_path}")
print("You can now submit this file to the competition!")

Submission validation:
Expected columns: ['id', 'x', 'y']
Our columns: ['id', 'x', 'y']
Expected rows: 5837
Our rows: 5837
Missing IDs: 0
Extra IDs: 0

Final submission shape: (5837, 3)
Expected shape: (5837, 3)
Shapes match: True
Missing values: 0

Submission file saved to: c:\nfl-big-data-bowl-2026-prediction\submission.csv
File size: 333.9 KB

Submission Summary:
Total predictions: 5837
X coordinate range: 2.4 to 119.0
Y coordinate range: 1.8 to 51.4
Average X: 59.8
Average Y: 27.2

🏈 SUBMISSION READY FOR LEADERBOARD! 🏈
File: c:\nfl-big-data-bowl-2026-prediction\submission.csv
You can now submit this file to the competition!
